# Genomic Feature Selection for RFS Prediction (Leakage-Free)

## Overview

This notebook performs genomic feature selection for a multimodal breast cancer Recurrence-Free Survival (RFS) prediction model using the TCGA-BRCA dataset.

The objective is to reduce the number of genomic features while retaining genes that may be informative for predicting recurrence-free survival. The selected genes are intended for use in the downstream multimodal Transformer survival model.

To prevent data leakage, **all feature-selection steps and standardization are fitted using training data only**. The same selected genes and fitted scaler are then applied to the validation and test datasets.

## Pipeline

1. Load the corrected RFS training, validation, and test splits, the audited RFS endpoint, and the previously preprocessed 1,000-gene datasets.
2. Normalize patient IDs and verify patient uniqueness, split membership, and consistency of survival times and event indicators.
3. Remove constant genomic features using a variance threshold of zero, fitted on the training data.
4. Remove highly correlated genes using Pearson correlation filtering with an absolute correlation threshold greater than 0.95, calculated on the training data.
5. Fit univariate Cox proportional hazards models for the remaining genes using training survival times and event indicators.
6. Rank genes by their Cox model p-values and select the top 200 genes.
7. Apply the same 200 selected genes to the training, validation, and test datasets.
8. Standardize the selected genes using `StandardScaler`, fitted only on the training data.
9. Perform final quality checks and save the selected datasets, gene list, and fitted scaler.

## Final Dataset Structure

Each output dataset contains:

- Patient ID (`patient_id`)
- 200 selected genomic features
- RFS survival time (`survival_time`)
- RFS event indicator (`event`)

The clinical and pathology features are not selected in this notebook; they are handled separately and combined with the genomic features during the multimodal dataset preparation stage.

## Leakage Prevention and Validation

The notebook includes checks to verify:

- Patient IDs are unique and consistent across datasets.
- Survival times and event indicators match the authoritative RFS endpoint.
- Training, validation, and test patients do not overlap.
- The same 200 genomic features and column order are used across all splits.
- The selected genomic values contain no missing or non-finite values.
- Standardization is fitted on training data and applied unchanged to validation and test data.
- The selected gene list and fitted scaler are saved for reproducibility.

## Output Files

The notebook saves the following files in `rfs_feature_selection_leakage_free_endpoint_v2`:

- `genomic_train_200_rfs_leakage_free.csv`
- `genomic_validation_200_rfs_leakage_free.csv`
- `genomic_test_200_rfs_leakage_free.csv`
- `rfs_selected_genes_200.csv`
- `genomic_standard_scaler_200.pkl`

The resulting datasets are prepared for integration into the downstream multimodal RFS survival prediction pipeline.

##Imports

In [1]:
# ============================================================
# IMPORTS
# ============================================================

import os
import pandas as pd
import numpy as np
!pip install lifelines

from sklearn.feature_selection import (
    VarianceThreshold
)
from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler


import warnings
warnings.filterwarnings("ignore")


print("Libraries imported successfully.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 8.6 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4120 sha256=819509effa85fa9e3a3a7102738df9406efb8911f9727352d64cd2f1fabc85ac
  Stored in directory: /root/.cache/pip/wheels/7e/16/46/9477f188924292d3bf1fb8fb42844201591abfc19b7ba6d868
Successfully built autograd-gamma
Libraries imported successfully.


##Mount Google Drive

In [2]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


##Define Paths

In [24]:

# ============================================================
# PROJECT PATHS — CORRECTED RFS ENDPOINT VERSION
# ============================================================

import os
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler

BASE_DIR = "/content/drive/MyDrive/TCGA_BRCA"

# Corrected split files: 236 train, 49 validation, 52 test
DATASET_DIR = os.path.join(
    BASE_DIR,
    "dataset_split_rfs_endpoint_v2"
)

# Existing leakage-free 1000-gene inputs
GENOMIC_DIR = os.path.join(
    BASE_DIR,
    "genomic_preprocessing_leakage_free"
)

# NEW output folder — preserves your previous feature-selection results
SAVE_DIR = os.path.join(
    BASE_DIR,
    "rfs_feature_selection_leakage_free_endpoint_v2"
)
os.makedirs(SAVE_DIR, exist_ok=True)

# Final authoritative RFS endpoint
ENDPOINT_PATH = (
    "/content/drive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

TRAIN_RFS_PATH = os.path.join(DATASET_DIR, "train_rfs.csv")
VAL_RFS_PATH = os.path.join(DATASET_DIR, "validation_rfs.csv")
TEST_RFS_PATH = os.path.join(DATASET_DIR, "test_rfs.csv")

TRAIN_GENOMIC_PATH = os.path.join(
    GENOMIC_DIR, "genomic_train_1000_leakage_free.csv"
)
VAL_GENOMIC_PATH = os.path.join(
    GENOMIC_DIR, "genomic_validation_1000_leakage_free.csv"
)
TEST_GENOMIC_PATH = os.path.join(
    GENOMIC_DIR, "genomic_test_1000_leakage_free.csv"
)

print("Paths configured.")
print("Corrected split directory:", DATASET_DIR)
print("New feature-selection output:", SAVE_DIR)


Paths configured.
Corrected split directory: /content/drive/MyDrive/TCGA_BRCA/dataset_split_rfs_endpoint_v2
New feature-selection output: /content/drive/MyDrive/TCGA_BRCA/rfs_feature_selection_leakage_free_endpoint_v2


##Load Dataset Splits

In [25]:

# ============================================================
# LOAD CORRECTED RFS SPLITS, ENDPOINT, AND GENOMIC DATA
# ============================================================

train_rfs = pd.read_csv(TRAIN_RFS_PATH)
validation_rfs = pd.read_csv(VAL_RFS_PATH)
test_rfs = pd.read_csv(TEST_RFS_PATH)

train_genomic = pd.read_csv(TRAIN_GENOMIC_PATH)
validation_genomic = pd.read_csv(VAL_GENOMIC_PATH)
test_genomic = pd.read_csv(TEST_GENOMIC_PATH)

endpoint = pd.read_csv(ENDPOINT_PATH, low_memory=False)

def normalize_patient_ids(df):
    df = df.copy()
    df["patient_id"] = (
        df["patient_id"].astype("string").str.strip().str.upper()
    )
    return df

train_rfs = normalize_patient_ids(train_rfs)
validation_rfs = normalize_patient_ids(validation_rfs)
test_rfs = normalize_patient_ids(test_rfs)

train_genomic = normalize_patient_ids(train_genomic)
validation_genomic = normalize_patient_ids(validation_genomic)
test_genomic = normalize_patient_ids(test_genomic)

endpoint = normalize_patient_ids(endpoint)

for name, df in {
    "train split": train_rfs,
    "validation split": validation_rfs,
    "test split": test_rfs,
    "endpoint": endpoint,
    "train genomic": train_genomic,
    "validation genomic": validation_genomic,
    "test genomic": test_genomic,
}.items():
    assert df["patient_id"].notna().all(), f"{name}: missing patient IDs"
    assert df["patient_id"].is_unique, f"{name}: duplicate patient IDs"

# Check the authoritative endpoint
assert {"patient_id", "survival_time", "event"}.issubset(endpoint.columns)
assert endpoint["survival_time"].notna().all()
assert (pd.to_numeric(endpoint["survival_time"]) > 0).all()
assert endpoint["event"].isin([0, 1]).all()

print("Corrected split sizes:")
print("Train:", len(train_rfs))
print("Validation:", len(validation_rfs))
print("Test:", len(test_rfs))

print("\nExisting genomic sizes before filtering:")
print("Train:", len(train_genomic))
print("Validation:", len(validation_genomic))
print("Test:", len(test_genomic))


Corrected split sizes:
Train: 236
Validation: 49
Test: 52

Existing genomic sizes before filtering:
Train: 249
Validation: 54
Test: 54


##CREATE THE DATA USED FOR FEATURE SELECTION

In [26]:

# ============================================================
# FILTER GENOMIC DATA AND ATTACH AUTHORITATIVE RFS OUTCOMES
# ============================================================

outcome_cols = ["patient_id", "survival_time", "event"]

endpoint_outcomes = endpoint[outcome_cols].copy()
endpoint_outcomes["survival_time"] = pd.to_numeric(
    endpoint_outcomes["survival_time"], errors="coerce"
)
endpoint_outcomes["event"] = pd.to_numeric(
    endpoint_outcomes["event"], errors="coerce"
)

assert endpoint_outcomes["patient_id"].is_unique
assert endpoint_outcomes["survival_time"].notna().all()
assert (endpoint_outcomes["survival_time"] > 0).all()
assert endpoint_outcomes["event"].isin([0, 1]).all()

def prepare_split(rfs_df, genomic_df, split_name):
    # Split membership comes from the corrected split file
    wanted_ids = rfs_df["patient_id"].tolist()
    wanted_set = set(wanted_ids)

    genomic_ids = set(genomic_df["patient_id"])

    missing_genomic = wanted_set - genomic_ids
    if missing_genomic:
        raise ValueError(
            f"{split_name}: {len(missing_genomic)} corrected patients "
            f"are missing from the genomic features. Examples: "
            f"{sorted(missing_genomic)[:10]}"
        )

    # Keep only patients belonging to this corrected split
    filtered = genomic_df[
        genomic_df["patient_id"].isin(wanted_set)
    ].copy()

    # Ensure each row follows the corrected split's patient order
    filtered = (
        filtered.set_index("patient_id")
        .loc[wanted_ids]
        .reset_index()
    )

    # Remove any outcome columns from the genomic source.
    # Outcomes will be attached afresh from the authoritative endpoint.
    features = [
        c for c in filtered.columns
        if c not in ["patient_id", "survival_time", "event"]
    ]

    filtered = filtered[["patient_id"] + features]

    # Attach authoritative survival times and event indicators
    filtered = filtered.merge(
        endpoint_outcomes,
        on="patient_id",
        how="left",
        validate="one_to_one"
    )

    assert len(filtered) == len(rfs_df)
    assert filtered["patient_id"].tolist() == wanted_ids
    assert filtered["survival_time"].notna().all()
    assert filtered["event"].notna().all()

    # Verify outcomes match the corrected split file
    expected = rfs_df[
        ["patient_id", "survival_time", "event"]
    ].copy()

    expected["survival_time"] = pd.to_numeric(
        expected["survival_time"], errors="coerce"
    )
    expected["event"] = pd.to_numeric(
        expected["event"], errors="coerce"
    )

    check = filtered[outcome_cols].merge(
        expected,
        on="patient_id",
        suffixes=("_endpoint", "_split"),
        validate="one_to_one"
    )

    assert np.allclose(
        check["survival_time_endpoint"],
        check["survival_time_split"]
    ), f"{split_name}: survival-time mismatch"

    assert (
        check["event_endpoint"].to_numpy()
        == check["event_split"].to_numpy()
    ).all(), f"{split_name}: event mismatch"

    print(
        f"{split_name}: {len(filtered)} patients; "
        f"{int(filtered['event'].sum())} events; "
        f"outcomes verified"
    )

    return filtered, features


train_genomic, genomic_features = prepare_split(
    train_rfs, train_genomic, "Train"
)

validation_genomic, validation_features = prepare_split(
    validation_rfs, validation_genomic, "Validation"
)

test_genomic, test_features = prepare_split(
    test_rfs, test_genomic, "Test"
)

# All three splits must have the same genomic feature columns
assert genomic_features == validation_features
assert genomic_features == test_features

print("\nPrepared genomic data:")
print("Train:", train_genomic.shape)
print("Validation:", validation_genomic.shape)
print("Test:", test_genomic.shape)
print("Genomic features:", len(genomic_features))


Train: 236 patients; 54 events; outcomes verified
Validation: 49 patients; 12 events; outcomes verified
Test: 52 patients; 12 events; outcomes verified

Prepared genomic data:
Train: (236, 1003)
Validation: (49, 1003)
Test: (52, 1003)
Genomic features: 1000


In [6]:
print(train_genomic.columns[-10:].tolist())
print("Outcome columns in genomic data:",
      [c for c in ["survival_time", "event"]
       if c in train_genomic.columns])

['SLURP1', 'LGALS7', 'EDN2', 'CT62', 'CST6', 'LRRN1', 'GFAP', 'ZNF98', 'MAG', 'ADRB1']
Outcome columns in genomic data: []


##Dataset Quality Control

In [27]:
# ============================================================
# QUALITY CONTROL
# ============================================================

print("="*70)
print("QUALITY CONTROL")
print("="*70)


datasets = {
    "Train": train_genomic,
    "Validation": validation_genomic,
    "Test": test_genomic
}


for name, df in datasets.items():

    print("="*70)
    print(name)

    print(
        "Patients:",
        df["patient_id"].nunique()
    )

    print(
        "Rows:",
        len(df)
    )

    print(
        "Duplicates:",
        df["patient_id"].duplicated().sum()
    )

    print(
        "Missing values:",
        df.isna().sum().sum()
    )

    print(
        "Number of genomic features:",
        len(df.columns) - 1
    )

    print()


# ------------------------------------------------------------
# Check RFS outcome information
# ------------------------------------------------------------

print("="*70)
print("RFS OUTCOME CHECK")
print("="*70)


for name, df in {
    "Train": train_outcome,
    "Validation": validation_outcome,
    "Test": test_outcome
}.items():

    print(
        name,
        "patients:",
        len(df)
    )

    print(
        name,
        "events:",
        df["event"].sum()
    )

    print()

QUALITY CONTROL
Train
Patients: 236
Rows: 236
Duplicates: 0
Missing values: 0
Number of genomic features: 1002

Validation
Patients: 49
Rows: 49
Duplicates: 0
Missing values: 0
Number of genomic features: 1002

Test
Patients: 52
Rows: 52
Duplicates: 0
Missing values: 0
Number of genomic features: 1002

RFS OUTCOME CHECK
Train patients: 249
Train events: 63

Validation patients: 54
Validation events: 14

Test patients: 54
Test events: 14



##Remove Constant Genes

In [28]:
# ============================================================
# REMOVE CONSTANT GENOMIC FEATURES
# ============================================================

# Variance filtering is fitted ONLY on the training dataset.
# Validation and test datasets are not used to decide which
# genomic features are removed.

X_train_genes = train_genomic[
    genomic_features
]


variance_selector = VarianceThreshold(
    threshold=0
)


X_train_variance = variance_selector.fit_transform(
    X_train_genes
)


selected_after_variance = (
    X_train_genes.columns[
        variance_selector.get_support()
    ]
)


print("="*70)
print("VARIANCE FILTER")
print("="*70)


print(
    "Original genes:",
    len(genomic_features)
)


print(
    "Remaining genes:",
    len(selected_after_variance)
)


print(
    "Removed genes:",
    len(genomic_features)
    -
    len(selected_after_variance)
)

VARIANCE FILTER
Original genes: 1000
Remaining genes: 1000
Removed genes: 0


##Remove Highly Correlated Genes

In [29]:
# ============================================================
# REMOVE HIGHLY CORRELATED GENES
# ============================================================

# Correlation filtering is performed ONLY on the training data.
# Validation and test data are not used for feature selection.

corr_matrix = (
    train_genomic[
        selected_after_variance
    ]
    .corr()
)


upper_triangle = (
    corr_matrix
    .where(
        np.triu(
            np.ones(
                corr_matrix.shape
            ),
            k=1
        ).astype(bool)
    )
)


correlated_features = [
    column
    for column in upper_triangle.columns
    if any(
        upper_triangle[column].abs() > 0.95
    )
]


genes_after_corr = [
    g
    for g in selected_after_variance
    if g not in correlated_features
]


print("="*70)
print("CORRELATION FILTER")
print("="*70)


print(
    "Genes before correlation filtering:",
    len(selected_after_variance)
)


print(
    "Removed highly correlated genes:",
    len(correlated_features)
)


print(
    "Remaining genes:",
    len(genes_after_corr)
)

CORRELATION FILTER
Genes before correlation filtering: 1000
Removed highly correlated genes: 6
Remaining genes: 994


##re build

In [34]:

# ============================================================
# REBUILD TRAINING OUTCOMES FROM THE AUTHORITATIVE RFS ENDPOINT
# ============================================================

import os
import pandas as pd

ENDPOINT_PATH = (
    "/content/drive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

# Load the authoritative endpoint
endpoint = pd.read_csv(ENDPOINT_PATH)

# Normalize patient IDs
endpoint["patient_id"] = (
    endpoint["patient_id"].astype(str).str.strip().str.upper()
)

train_genomic["patient_id"] = (
    train_genomic["patient_id"].astype(str).str.strip().str.upper()
)

# Confirm endpoint integrity
assert endpoint["patient_id"].is_unique, (
    "Duplicate patient IDs in the authoritative endpoint."
)

assert {"patient_id", "survival_time", "event"}.issubset(
    endpoint.columns
), "The endpoint is missing required columns."

# Keep only the patients in the corrected training genomic dataset
train_outcome = train_genomic[["patient_id"]].merge(
    endpoint[["patient_id", "survival_time", "event"]],
    on="patient_id",
    how="left",
    validate="one_to_one",
    sort=False
)

# Verify that every training patient has an endpoint
assert train_outcome["survival_time"].notna().all(), (
    "Some training patients have no survival time in the endpoint."
)

assert train_outcome["event"].notna().all(), (
    "Some training patients have no event value in the endpoint."
)

assert train_outcome["event"].isin([0, 1]).all()
assert (train_outcome["survival_time"] > 0).all()

# Verify patient order
assert train_outcome["patient_id"].equals(
    train_genomic["patient_id"]
)

print("TRAINING OUTCOMES REBUILT SUCCESSFULLY")
print("Genomic training patients:", len(train_genomic))
print("Outcome training patients:", len(train_outcome))
print("Training events:", int(train_outcome["event"].sum()))
print("Training censored:", int((train_outcome["event"] == 0).sum()))
print("Patient order matches:", train_outcome["patient_id"].equals(
    train_genomic["patient_id"]
))


TRAINING OUTCOMES REBUILT SUCCESSFULLY
Genomic training patients: 236
Outcome training patients: 236
Training events: 54
Training censored: 182
Patient order matches: True


##COX SURVIVAL Feature Selection

In [35]:

# ============================================================
# COX SURVIVAL FEATURE SELECTION — TRAINING DATA ONLY
# ============================================================

from lifelines import CoxPHFitter
import numpy as np
import pandas as pd

TOP_K = 200

cox_results = []
failed_genes = []

# ============================================================
# 1. CHECK PATIENT IDS
# ============================================================

# Convert IDs to a consistent format
train_genomic["patient_id"] = (
    train_genomic["patient_id"].astype(str).str.strip().str.upper()
)

train_outcome["patient_id"] = (
    train_outcome["patient_id"].astype(str).str.strip().str.upper()
)

# Check for duplicate patient IDs
assert train_genomic["patient_id"].is_unique, (
    "Duplicate patient IDs in train_genomic."
)

assert train_outcome["patient_id"].is_unique, (
    "Duplicate patient IDs in train_outcome."
)

genomic_ids = set(train_genomic["patient_id"])
outcome_ids = set(train_outcome["patient_id"])

print("Genomic patients:", len(genomic_ids))
print("Outcome patients:", len(outcome_ids))
print("Patients missing outcomes:", len(genomic_ids - outcome_ids))
print("Extra patients in outcomes:", len(outcome_ids - genomic_ids))

# Patient sets must match exactly
if genomic_ids != outcome_ids:
    raise ValueError(
        "Patient IDs do not match between genomic data and outcomes. "
        "Stop and investigate before Cox feature selection."
    )

# ============================================================
# 2. ALIGN OUTCOMES WITH GENOMIC ROW ORDER
# ============================================================

train_outcome = train_genomic[["patient_id"]].merge(
    train_outcome[["patient_id", "survival_time", "event"]],
    on="patient_id",
    how="left",
    validate="one_to_one",
    sort=False
)

# Confirm the IDs now match in both dataframes
assert train_outcome["patient_id"].equals(
    train_genomic["patient_id"]
)

# ============================================================
# 3. VALIDATE SURVIVAL OUTCOMES
# ============================================================

train_outcome["survival_time"] = pd.to_numeric(
    train_outcome["survival_time"], errors="coerce"
)

train_outcome["event"] = pd.to_numeric(
    train_outcome["event"], errors="coerce"
)

assert train_outcome["survival_time"].notna().all(), (
    "Missing or invalid survival times."
)

assert train_outcome["event"].notna().all(), (
    "Missing or invalid event values."
)

assert np.isfinite(
    train_outcome[["survival_time", "event"]].to_numpy()
).all(), "Non-finite survival outcomes."

assert (train_outcome["survival_time"] > 0).all(), (
    "Survival times must be greater than zero."
)

assert train_outcome["event"].isin([0, 1]).all(), (
    "Event values must be 0 or 1."
)

print("PASS: Patient IDs and survival outcomes are aligned.")
print("Training patients:", len(train_outcome))
print("Observed training events:", int(train_outcome["event"].sum()))

# ============================================================
# 4. COX FEATURE SELECTION — TRAINING DATA ONLY
# ============================================================

for gene in genes_after_corr:

    temp = pd.DataFrame({
        gene: pd.to_numeric(
            train_genomic[gene], errors="coerce"
        ).to_numpy(),
        "survival_time": train_outcome[
            "survival_time"
        ].to_numpy(),
        "event": train_outcome["event"].to_numpy()
    })

    # Do not fit a Cox model to invalid values
    if not np.isfinite(temp.to_numpy()).all():
        failed_genes.append(
            (gene, "missing/non-finite values")
        )
        continue

    try:
        cph = CoxPHFitter()

        cph.fit(
            temp,
            duration_col="survival_time",
            event_col="event"
        )

        p_value = float(cph.summary.loc[gene, "p"])

        if np.isfinite(p_value):
            cox_results.append({
                "gene": gene,
                "p_value": p_value
            })
        else:
            failed_genes.append(
                (gene, "non-finite p-value")
            )

    except Exception as exc:
        failed_genes.append((gene, str(exc)))

# ============================================================
# 5. RANK GENES AND SELECT THE TOP 200
# ============================================================

cox_results = pd.DataFrame(
    cox_results,
    columns=["gene", "p_value"]
)

if cox_results.empty:
    raise RuntimeError(
        "No valid univariate Cox models succeeded."
    )

cox_results = cox_results.sort_values(
    "p_value"
).reset_index(drop=True)

selected_genes = (
    cox_results.head(TOP_K)["gene"].tolist()
)

print("=" * 60)
print("COX FEATURE SELECTION RESULTS")
print("=" * 60)
print("Training patients:", len(train_outcome))
print("Observed training events:", int(train_outcome["event"].sum()))
print("Genes entering Cox analysis:", len(genes_after_corr))
print("Successful Cox models:", len(cox_results))
print("Failed genes:", len(failed_genes))
print("Genes selected:", len(selected_genes))
print("Top 20 genes:", selected_genes[:20])

if len(selected_genes) != TOP_K:
    raise RuntimeError(
        f"Expected {TOP_K} genes but selected "
        f"{len(selected_genes)}. Investigate before proceeding."
    )

assert len(set(selected_genes)) == TOP_K
assert set(selected_genes).issubset(
    set(train_genomic.columns)
)

print("PASS: exactly 200 unique training-selected genes.")


Genomic patients: 236
Outcome patients: 236
Patients missing outcomes: 0
Extra patients in outcomes: 0
PASS: Patient IDs and survival outcomes are aligned.
Training patients: 236
Observed training events: 54
COX FEATURE SELECTION RESULTS
Training patients: 236
Observed training events: 54
Genes entering Cox analysis: 994
Successful Cox models: 994
Failed genes: 0
Genes selected: 200
Top 20 genes: ['PSCA', 'ZNF385B', 'MYBPC1', 'CALML5', 'PRAME', 'FLT3', 'GREB1', 'TSPAN8', 'IL6ST', 'TUBA3E', 'RP1', 'DUSP9', 'S100P', 'LRRN1', 'FAM83A', 'CACNG4', 'LOC100128977', 'C1orf173', 'LOC100130148', 'HOTAIR']
PASS: exactly 200 unique training-selected genes.


##Apply Selection to All Splits

In [36]:
# ============================================================
# APPLY FEATURE SELECTION
# ============================================================

# The 200 genes were selected using TRAINING data only.
# The exact same genes are now applied to validation and test.
#
# No feature selection is performed on validation or test data.


# ------------------------------------------------------------
# Select the 200 genes from each dataset
# ------------------------------------------------------------

train_selected = train_genomic[
    [
        "patient_id"
    ] + selected_genes
].copy()


validation_selected = validation_genomic[
    [
        "patient_id"
    ] + selected_genes
].copy()


test_selected = test_genomic[
    [
        "patient_id"
    ] + selected_genes
].copy()


# ------------------------------------------------------------
# Add RFS outcome information
# ------------------------------------------------------------

train_selected = train_selected.merge(
    train_outcome,
    on="patient_id",
    how="left",
    validate="one_to_one"
)

validation_selected = validation_selected.merge(
    validation_outcome,
    on="patient_id",
    how="left",
    validate="one_to_one"
)

test_selected = test_selected.merge(
    test_outcome,
    on="patient_id",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# Reorder columns
# ------------------------------------------------------------

final_columns = (
    [
        "patient_id"
    ]
    +
    selected_genes
    +
    [
        "survival_time",
        "event"
    ]
)


train_selected = train_selected[
    final_columns
]

validation_selected = validation_selected[
    final_columns
]

test_selected = test_selected[
    final_columns
]


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("="*70)
print("FEATURE SELECTION APPLIED")
print("="*70)


print(
    "Train:",
    train_selected.shape
)

print(
    "Validation:",
    validation_selected.shape
)

print(
    "Test:",
    test_selected.shape
)


print(
    "\nSelected genomic genes:",
    len(selected_genes)
)

FEATURE SELECTION APPLIED
Train: (236, 203)
Validation: (49, 203)
Test: (52, 203)

Selected genomic genes: 200


##FINAL FEATURE SELECTION QC

In [37]:
# ============================================================
# FINAL FEATURE SELECTION QC
# ============================================================

print("="*70)
print("FINAL FEATURE SELECTION QC")
print("="*70)


# ------------------------------------------------------------
# 1. Check patient counts
# ------------------------------------------------------------

print("\nPATIENT COUNTS")

print(
    "Train:",
    train_selected["patient_id"].nunique()
)

print(
    "Validation:",
    validation_selected["patient_id"].nunique()
)

print(
    "Test:",
    test_selected["patient_id"].nunique()
)


# ------------------------------------------------------------
# 2. Check duplicate patients
# ------------------------------------------------------------

print("\nDUPLICATE PATIENTS")

print(
    "Train:",
    train_selected["patient_id"].duplicated().sum()
)

print(
    "Validation:",
    validation_selected["patient_id"].duplicated().sum()
)

print(
    "Test:",
    test_selected["patient_id"].duplicated().sum()
)


# ------------------------------------------------------------
# 3. Check missing values
# ------------------------------------------------------------

print("\nMISSING VALUES")

print(
    "Train:",
    train_selected.isna().sum().sum()
)

print(
    "Validation:",
    validation_selected.isna().sum().sum()
)

print(
    "Test:",
    test_selected.isna().sum().sum()
)


# ------------------------------------------------------------
# 4. Check number of genomic features
# ------------------------------------------------------------

print("\nGENOMIC FEATURES")

print(
    "Train:",
    len(
        [
            c for c in train_selected.columns
            if c not in [
                "patient_id",
                "survival_time",
                "event"
            ]
        ]
    )
)

print(
    "Validation:",
    len(
        [
            c for c in validation_selected.columns
            if c not in [
                "patient_id",
                "survival_time",
                "event"
            ]
        ]
    )
)

print(
    "Test:",
    len(
        [
            c for c in test_selected.columns
            if c not in [
                "patient_id",
                "survival_time",
                "event"
            ]
        ]
    )
)


# ------------------------------------------------------------
# 5. Check column consistency
# ------------------------------------------------------------

print("\nCOLUMN CONSISTENCY")

print(
    "Train == Validation:",
    train_selected.columns.equals(
        validation_selected.columns
    )
)

print(
    "Validation == Test:",
    validation_selected.columns.equals(
        test_selected.columns
    )
)


# ------------------------------------------------------------
# 6. Check RFS outcomes
# ------------------------------------------------------------

print("\nRFS EVENTS")

print(
    "Train events:",
    train_selected["event"].sum()
)

print(
    "Validation events:",
    validation_selected["event"].sum()
)

print(
    "Test events:",
    test_selected["event"].sum()
)


# ------------------------------------------------------------
# 7. Check survival time
# ------------------------------------------------------------

print("\nSURVIVAL TIME")

print(
    "Train minimum:",
    train_selected["survival_time"].min()
)

print(
    "Validation minimum:",
    validation_selected["survival_time"].min()
)

print(
    "Test minimum:",
    test_selected["survival_time"].min()
)

FINAL FEATURE SELECTION QC

PATIENT COUNTS
Train: 236
Validation: 49
Test: 52

DUPLICATE PATIENTS
Train: 0
Validation: 0
Test: 0

MISSING VALUES
Train: 0
Validation: 0
Test: 0

GENOMIC FEATURES
Train: 200
Validation: 200
Test: 200

COLUMN CONSISTENCY
Train == Validation: True
Validation == Test: True

RFS EVENTS
Train events: 54
Validation events: 12
Test events: 12

SURVIVAL TIME
Train minimum: 1.0
Validation minimum: 10.0
Test minimum: 10.0


##Scale the final 200 genes

In [38]:
# ============================================================
# STANDARDIZE FINAL 200 GENOMIC FEATURES
# ============================================================

from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Define the final genomic feature list
# ------------------------------------------------------------

final_genomic_features = selected_genes


# ------------------------------------------------------------
# Create scaler
# ------------------------------------------------------------

genomic_scaler = StandardScaler()


# ------------------------------------------------------------
# FIT SCALER ONLY ON TRAINING DATA
# ------------------------------------------------------------

train_selected[
    final_genomic_features
] = genomic_scaler.fit_transform(
    train_selected[
        final_genomic_features
    ]
)


# ------------------------------------------------------------
# TRANSFORM VALIDATION AND TEST
# ------------------------------------------------------------

validation_selected[
    final_genomic_features
] = genomic_scaler.transform(
    validation_selected[
        final_genomic_features
    ]
)


test_selected[
    final_genomic_features
] = genomic_scaler.transform(
    test_selected[
        final_genomic_features
    ]
)


print("="*70)
print("GENOMIC STANDARDIZATION")
print("="*70)


print(
    "Scaler fitted on:",
    len(train_selected),
    "training patients"
)


print(
    "Features standardized:",
    len(final_genomic_features)
)


print(
    "Validation transformed using training scaler: YES"
)


print(
    "Test transformed using training scaler: YES"
)

GENOMIC STANDARDIZATION
Scaler fitted on: 236 training patients
Features standardized: 200
Validation transformed using training scaler: YES
Test transformed using training scaler: YES


In [39]:
# ============================================================
# CHECK STANDARDIZATION
# ============================================================

print("="*70)
print("STANDARDIZATION CHECK")
print("="*70)


train_means = (
    train_selected[
        final_genomic_features
    ].mean()
)


train_stds = (
    train_selected[
        final_genomic_features
    ].std()
)


print(
    "Mean of training feature means:",
    train_means.mean()
)


print(
    "Mean of training feature stds:",
    train_stds.mean()
)


print(
    "Maximum absolute training mean:",
    train_means.abs().max()
)


print(
    "Minimum training std:",
    train_stds.min()
)


print(
    "\nExample standardized values:"
)


print(
    train_selected[
        final_genomic_features[:5]
    ].head()
)

STANDARDIZATION CHECK
Mean of training feature means: 5.4477373647946905e-17
Mean of training feature stds: 1.0021254009099538
Maximum absolute training mean: 8.354898693789314e-16
Minimum training std: 1.0021254009099532

Example standardized values:
       PSCA   ZNF385B    MYBPC1    CALML5     PRAME
0  0.403351 -1.397859  0.612464  1.597367  1.174156
1  0.489042  0.458672 -0.382434  1.540094 -0.868016
2  0.764049 -0.645411 -0.627576  0.914746 -0.015223
3  0.393212 -0.380981 -1.165068  1.478743 -0.770307
4  0.830510 -0.371746  0.342282  1.336098  1.664718


##Checking cell

In [40]:

# ============================================================
# FINAL SAFETY CHECK — 200-GENE DATASETS
# ============================================================

assert len(selected_genes) == 200
assert len(set(selected_genes)) == 200

for name, df in {
    "train": train_selected,
    "validation": validation_selected,
    "test": test_selected
}.items():

    expected_columns = (
        ["patient_id"]
        + selected_genes
        + ["survival_time", "event"]
    )

    assert df.columns.tolist() == expected_columns, (
        f"{name}: unexpected column order or columns"
    )

    assert df["patient_id"].is_unique, (
        f"{name}: duplicate patient IDs"
    )

    assert df["survival_time"].notna().all()
    assert df["event"].notna().all()
    assert (df["survival_time"] > 0).all()
    assert df["event"].isin([0, 1]).all()

    assert np.isfinite(
        df[selected_genes].to_numpy(dtype=float)
    ).all(), f"{name}: non-finite genomic values"

assert set(train_selected["patient_id"]).isdisjoint(
    set(validation_selected["patient_id"])
)

assert set(train_selected["patient_id"]).isdisjoint(
    set(test_selected["patient_id"])
)

assert set(validation_selected["patient_id"]).isdisjoint(
    set(test_selected["patient_id"])
)

assert train_selected.columns.equals(
    validation_selected.columns
)

assert validation_selected.columns.equals(
    test_selected.columns
)

print("PASS: 200-gene datasets passed all final checks.")
print("Train:", train_selected.shape)
print("Validation:", validation_selected.shape)
print("Test:", test_selected.shape)


PASS: 200-gene datasets passed all final checks.
Train: (236, 203)
Validation: (49, 203)
Test: (52, 203)


##Save DataSet

In [41]:
# ============================================================
# SAVE FINAL LEAKAGE-FREE GENOMIC DATASETS
# ============================================================

train_selected.to_csv(
    os.path.join(
        SAVE_DIR,
        "genomic_train_200_rfs_leakage_free.csv"
    ),
    index=False
)


validation_selected.to_csv(
    os.path.join(
        SAVE_DIR,
        "genomic_validation_200_rfs_leakage_free.csv"
    ),
    index=False
)


test_selected.to_csv(
    os.path.join(
        SAVE_DIR,
        "genomic_test_200_rfs_leakage_free.csv"
    ),
    index=False
)


print("="*70)
print("FINAL GENOMIC DATASETS SAVED")
print("="*70)


print(
    "Train:",
    train_selected.shape
)

print(
    "Validation:",
    validation_selected.shape
)

print(
    "Test:",
    test_selected.shape
)

FINAL GENOMIC DATASETS SAVED
Train: (236, 203)
Validation: (49, 203)
Test: (52, 203)


##save the scaler

In [42]:
# ============================================================
# SAVE GENOMIC SCALER
# ============================================================

import joblib


SCALER_PATH = os.path.join(
    SAVE_DIR,
    "genomic_standard_scaler_200.pkl"
)


joblib.dump(
    genomic_scaler,
    SCALER_PATH
)


print(
    "Genomic scaler saved:"
)

print(
    SCALER_PATH
)

Genomic scaler saved:
/content/drive/MyDrive/TCGA_BRCA/rfs_feature_selection_leakage_free_endpoint_v2/genomic_standard_scaler_200.pkl


##save genes list:

In [43]:
# ============================================================
# SAVE FINAL 200-GENE LIST
# ============================================================

GENE_LIST_PATH = os.path.join(
    SAVE_DIR,
    "rfs_selected_genes_200.csv"
)

pd.DataFrame({
    "rfs_selected_genes": selected_genes
}).to_csv(
    GENE_LIST_PATH,
    index=False
)

print("Final 200-gene list saved:")
print(GENE_LIST_PATH)

print("\nNumber of genes saved:", len(selected_genes))

Final 200-gene list saved:
/content/drive/MyDrive/TCGA_BRCA/rfs_feature_selection_leakage_free_endpoint_v2/rfs_selected_genes_200.csv

Number of genes saved: 200


##Final Feature Selection Verification

In [44]:
# ============================================================
# FINAL FEATURE SELECTION VERIFICATION
# ============================================================

print("="*70)
print("FINAL RFS FEATURE SELECTION VERIFICATION")
print("="*70)

# ------------------------------------------------------------
# 1. Dataset shapes
# ------------------------------------------------------------

print("\nDATASET SHAPES")
print("Train:", train_selected.shape)
print("Validation:", validation_selected.shape)
print("Test:", test_selected.shape)

# ------------------------------------------------------------
# 2. Patient counts
# ------------------------------------------------------------

print("\nPATIENT COUNTS")
print("Train:", train_selected["patient_id"].nunique())
print("Validation:", validation_selected["patient_id"].nunique())
print("Test:", test_selected["patient_id"].nunique())

# ------------------------------------------------------------
# 3. Duplicate patients
# ------------------------------------------------------------

print("\nDUPLICATE PATIENTS")
print("Train:", train_selected["patient_id"].duplicated().sum())
print("Validation:", validation_selected["patient_id"].duplicated().sum())
print("Test:", test_selected["patient_id"].duplicated().sum())

# ------------------------------------------------------------
# 4. Missing values
# ------------------------------------------------------------

print("\nMISSING VALUES")
print("Train:", train_selected.isna().sum().sum())
print("Validation:", validation_selected.isna().sum().sum())
print("Test:", test_selected.isna().sum().sum())

# ------------------------------------------------------------
# 5. Genomic feature count
# ------------------------------------------------------------

feature_columns = [
    c for c in train_selected.columns
    if c not in ["patient_id", "survival_time", "event"]
]

print("\nGENOMIC FEATURES")
print("Number of selected genes:", len(feature_columns))

# ------------------------------------------------------------
# 6. Column consistency
# ------------------------------------------------------------

print("\nCOLUMN CONSISTENCY")
print(
    "Train == Validation:",
    train_selected.columns.equals(validation_selected.columns)
)
print(
    "Validation == Test:",
    validation_selected.columns.equals(test_selected.columns)
)

# ------------------------------------------------------------
# 7. RFS events
# ------------------------------------------------------------

print("\nRFS EVENTS")
print("Train events:", int(train_selected["event"].sum()))
print("Validation events:", int(validation_selected["event"].sum()))
print("Test events:", int(test_selected["event"].sum()))

# ------------------------------------------------------------
# 8. Standardization check
# ------------------------------------------------------------

train_means = train_selected[feature_columns].mean()
train_stds = train_selected[feature_columns].std()

print("\nSTANDARDIZATION")
print("Maximum absolute training mean:", train_means.abs().max())
print("Minimum training standard deviation:", train_stds.min())
print("Maximum training standard deviation:", train_stds.max())

# ------------------------------------------------------------
# 9. Saved files
# ------------------------------------------------------------

print("\nSAVED FILES")

expected_files = [
    "genomic_train_200_rfs_leakage_free.csv",
    "genomic_validation_200_rfs_leakage_free.csv",
    "genomic_test_200_rfs_leakage_free.csv",
    "rfs_selected_genes_200.csv",
    "genomic_standard_scaler_200.pkl"
]

for filename in expected_files:

    path = os.path.join(
        SAVE_DIR,
        filename
    )

    print(
        filename,
        "->",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

print("\n" + "="*70)
print("VERIFICATION COMPLETE")
print("="*70)

FINAL RFS FEATURE SELECTION VERIFICATION

DATASET SHAPES
Train: (236, 203)
Validation: (49, 203)
Test: (52, 203)

PATIENT COUNTS
Train: 236
Validation: 49
Test: 52

DUPLICATE PATIENTS
Train: 0
Validation: 0
Test: 0

MISSING VALUES
Train: 0
Validation: 0
Test: 0

GENOMIC FEATURES
Number of selected genes: 200

COLUMN CONSISTENCY
Train == Validation: True
Validation == Test: True

RFS EVENTS
Train events: 54
Validation events: 12
Test events: 12

STANDARDIZATION
Maximum absolute training mean: 8.354898693789314e-16
Minimum training standard deviation: 1.0021254009099532
Maximum training standard deviation: 1.0021254009099554

SAVED FILES
genomic_train_200_rfs_leakage_free.csv -> EXISTS
genomic_validation_200_rfs_leakage_free.csv -> EXISTS
genomic_test_200_rfs_leakage_free.csv -> EXISTS
rfs_selected_genes_200.csv -> EXISTS
genomic_standard_scaler_200.pkl -> EXISTS

VERIFICATION COMPLETE


In [45]:
# ============================================================
# VERIFY FINAL GENE LIST
# ============================================================

gene_list_check = pd.read_csv(
    GENE_LIST_PATH
)

print("Gene list exists:", os.path.exists(GENE_LIST_PATH))
print("Number of genes:", len(gene_list_check))
print("Unique genes:", gene_list_check["rfs_selected_genes"].nunique())

Gene list exists: True
Number of genes: 200
Unique genes: 200


In [46]:
# ============================================================
# AUDIT: FEATURE-SELECTION RFS SPLITS VS AUTHORITATIVE CANDIDATE
# READ-ONLY DIAGNOSTIC
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/TCGA_BRCA")

RFS_SPLIT_DIR = BASE_DIR / "dataset_split_rfs"

ENDPOINT_PATH = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

def clean_patient_id(series):
    return series.astype("string").str.strip().str.upper()

# Load the three split files and the selected endpoint
train_rfs_check = pd.read_csv(RFS_SPLIT_DIR / "train_rfs.csv")
val_rfs_check = pd.read_csv(RFS_SPLIT_DIR / "validation_rfs.csv")
test_rfs_check = pd.read_csv(RFS_SPLIT_DIR / "test_rfs.csv")
endpoint_check = pd.read_csv(ENDPOINT_PATH)

required = {"patient_id", "survival_time", "event"}
for name, df in {
    "train": train_rfs_check,
    "validation": val_rfs_check,
    "test": test_rfs_check,
    "endpoint": endpoint_check,
}.items():
    missing_cols = required - set(df.columns)
    assert not missing_cols, f"{name} is missing columns: {missing_cols}"

# Normalize IDs and outcomes
split_frames = {
    "train": train_rfs_check,
    "validation": val_rfs_check,
    "test": test_rfs_check,
}

for name, df in split_frames.items():
    df["patient_id"] = clean_patient_id(df["patient_id"])
    df["survival_time"] = pd.to_numeric(
        df["survival_time"], errors="coerce"
    )
    df["event"] = pd.to_numeric(df["event"], errors="coerce")

endpoint_check["patient_id"] = clean_patient_id(
    endpoint_check["patient_id"]
)
endpoint_check["survival_time"] = pd.to_numeric(
    endpoint_check["survival_time"], errors="coerce"
)
endpoint_check["event"] = pd.to_numeric(
    endpoint_check["event"], errors="coerce"
)

# Check for duplicate IDs and overlap across splits
for name, df in split_frames.items():
    print(f"{name}: {len(df)} rows; "
          f"{df['patient_id'].nunique()} unique IDs; "
          f"{df['patient_id'].isna().sum()} missing IDs")
    assert df["patient_id"].notna().all(), f"{name}: missing patient IDs"
    assert df["patient_id"].is_unique, f"{name}: duplicate patient IDs"

assert endpoint_check["patient_id"].notna().all()
assert endpoint_check["patient_id"].is_unique, (
    "The endpoint has duplicate patient IDs; inspect before proceeding."
)

split_ids = set().union(*[
    set(df["patient_id"]) for df in split_frames.values()
])
endpoint_ids = set(endpoint_check["patient_id"])

print("\n" + "=" * 70)
print("PATIENT-ID COMPARISON")
print("=" * 70)
print("Total patients across split files:", len(split_ids))
print("Patients in endpoint candidate:", len(endpoint_ids))
print("Split patients found in endpoint:", len(split_ids & endpoint_ids))
print("Split patients missing from endpoint:", len(split_ids - endpoint_ids))
print("Endpoint patients absent from splits:", len(endpoint_ids - split_ids))

missing_from_endpoint = sorted(split_ids - endpoint_ids)
extra_in_endpoint = sorted(endpoint_ids - split_ids)

print("\nSplit patients missing from endpoint:")
print(missing_from_endpoint)

print("\nEndpoint patients not assigned to these splits (first 20):")
print(extra_in_endpoint[:20])

# Compare outcomes for IDs present in both sources
split_all = pd.concat(
    [df[["patient_id", "survival_time", "event"]]
     for df in split_frames.values()],
    ignore_index=True
)

assert split_all["patient_id"].is_unique, (
    "Patient IDs overlap across train/validation/test files."
)

matched = split_all.merge(
    endpoint_check[["patient_id", "survival_time", "event"]],
    on="patient_id",
    how="inner",
    suffixes=("_split", "_endpoint"),
    validate="one_to_one"
)

time_equal = np.isclose(
    matched["survival_time_split"],
    matched["survival_time_endpoint"],
    equal_nan=False
)
event_equal = (
    matched["event_split"].to_numpy()
    == matched["event_endpoint"].to_numpy()
)

mismatches = matched.loc[
    (~time_equal) | (~event_equal)
].copy()

print("\n" + "=" * 70)
print("OUTCOME COMPARISON FOR MATCHED PATIENTS")
print("=" * 70)
print("Matched patients:", len(matched))
print("Survival/event disagreements:", len(mismatches))

if not mismatches.empty:
    display(mismatches.head(30))

# Show the split-file outcomes for the missing IDs
if missing_from_endpoint:
    missing_rows = split_all.loc[
        split_all["patient_id"].isin(missing_from_endpoint)
    ].sort_values("patient_id")

    print("\nOutcomes assigned to split patients absent from endpoint:")
    display(missing_rows)

print("\nAudit complete. No source files were modified.")

train: 249 rows; 249 unique IDs; 0 missing IDs
validation: 54 rows; 54 unique IDs; 0 missing IDs
test: 54 rows; 54 unique IDs; 0 missing IDs

PATIENT-ID COMPARISON
Total patients across split files: 357
Patients in endpoint candidate: 1062
Split patients found in endpoint: 337
Split patients missing from endpoint: 20
Endpoint patients absent from splits: 725

Split patients missing from endpoint:
['TCGA-A2-A04V', 'TCGA-A2-A0CM', 'TCGA-A2-A0SW', 'TCGA-A2-A0SX', 'TCGA-A8-A06N', 'TCGA-A8-A06Y', 'TCGA-A8-A08O', 'TCGA-A8-A08T', 'TCGA-A8-A08X', 'TCGA-A8-A091', 'TCGA-A8-A094', 'TCGA-A8-A096', 'TCGA-AC-A23H', 'TCGA-AC-A3W6', 'TCGA-AR-A0TW', 'TCGA-AR-A24O', 'TCGA-C8-A12Q', 'TCGA-C8-A12T', 'TCGA-C8-A26Y', 'TCGA-D8-A1XC']

Endpoint patients not assigned to these splits (first 20):
['TCGA-3C-AAAU', 'TCGA-3C-AALI', 'TCGA-3C-AALJ', 'TCGA-3C-AALK', 'TCGA-4H-AAAK', 'TCGA-5L-AAT0', 'TCGA-5T-A9QA', 'TCGA-A1-A0SD', 'TCGA-A1-A0SE', 'TCGA-A1-A0SG', 'TCGA-A1-A0SH', 'TCGA-A1-A0SK', 'TCGA-A1-A0SO', 'TCGA-A1-A

,patient_id,survival_time_split,event_split,survival_time_endpoint,event_endpoint
0,TCGA-A2-A3XU,912.0,1,574.0,1
3,TCGA-B6-A0WX,639.0,1,563.0,1
4,TCGA-B6-A0I8,749.0,1,746.0,1
5,TCGA-B6-A0WW,558.0,1,447.0,1
7,TCGA-E2-A14Z,563.0,1,188.0,1
13,TCGA-A2-A25B,1291.0,1,1290.0,1
18,TCGA-A2-A04P,548.0,1,102.0,1
28,TCGA-D8-A73W,385.0,1,21.0,1
37,TCGA-AO-A0JA,655.0,1,143.0,1
44,TCGA-BH-A1EY,538.0,1,396.0,1



Outcomes assigned to split patients absent from endpoint:


,patient_id,survival_time,event
15,TCGA-A2-A04V,1920.0,1
317,TCGA-A2-A0CM,754.0,1
48,TCGA-A2-A0SW,1365.0,1
0,TCGA-A2-A0SX,1534.0,1
224,TCGA-A8-A06N,1.0,0
287,TCGA-A8-A06Y,791.0,1
6,TCGA-A8-A08O,943.0,1
107,TCGA-A8-A08T,3409.0,1
265,TCGA-A8-A08X,1308.0,1
83,TCGA-A8-A091,1004.0,1



Audit complete. No source files were modified.


In [47]:
# ============================================================
# COMPARE ALL RFS VERSIONS AGAINST THE EXISTING SPLITS
# READ-ONLY: DOES NOT MODIFY ANY FILES
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

RFS_DIR = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint"
)

BASE_DIR = Path("/content/drive/MyDrive/TCGA_BRCA")
SPLIT_DIR = BASE_DIR / "dataset_split_rfs"

# Load the existing split files
split_frames = [
    pd.read_csv(SPLIT_DIR / "train_rfs.csv"),
    pd.read_csv(SPLIT_DIR / "validation_rfs.csv"),
    pd.read_csv(SPLIT_DIR / "test_rfs.csv"),
]

split_all = pd.concat(split_frames, ignore_index=True)

split_all["patient_id"] = (
    split_all["patient_id"].astype("string").str.strip().str.upper()
)
split_all["survival_time"] = pd.to_numeric(
    split_all["survival_time"], errors="coerce"
)
split_all["event"] = pd.to_numeric(
    split_all["event"], errors="coerce"
)

assert split_all["patient_id"].is_unique

versions = [
    "followup_rfs_audit_all_patients.csv",
    "followup_rfs_audited_v2_candidate.csv",
    "followup_rfs_final_v1.csv",
]

for filename in versions:
    path = RFS_DIR / filename
    df = pd.read_csv(path, low_memory=False)

    required = {"patient_id", "survival_time", "event"}
    if not required.issubset(df.columns):
        print(f"\n{filename}: REQUIRED COLUMNS MISSING")
        print("Columns:", df.columns.tolist())
        continue

    df["patient_id"] = (
        df["patient_id"].astype("string").str.strip().str.upper()
    )
    df["survival_time"] = pd.to_numeric(
        df["survival_time"], errors="coerce"
    )
    df["event"] = pd.to_numeric(df["event"], errors="coerce")

    print("\n" + "=" * 75)
    print("ENDPOINT VERSION:", filename)
    print("Rows:", len(df))
    print("Unique IDs:", df["patient_id"].nunique())

    if not df["patient_id"].is_unique:
        print("WARNING: Duplicate patient IDs; skipping outcome comparison.")
        continue

    comparison = split_all.merge(
        df[["patient_id", "survival_time", "event"]],
        on="patient_id",
        how="left",
        suffixes=("_split", "_version"),
        validate="one_to_one",
        indicator=True,
    )

    matched = comparison["_merge"].eq("both")
    time_match = np.isclose(
        comparison["survival_time_split"],
        comparison["survival_time_version"],
        equal_nan=False,
    )
    event_match = (
        comparison["event_split"].to_numpy()
        == comparison["event_version"].to_numpy()
    )

    disagreements = comparison.loc[
        matched & ((~time_match) | (~event_match))
    ].copy()

    missing = comparison.loc[
        ~matched, ["patient_id", "survival_time_split", "event_split"]
    ]

    print("Split patients:", len(split_all))
    print("Matched to this version:", int(matched.sum()))
    print("Missing from this version:", len(missing))
    print("Outcome disagreements among matched:", len(disagreements))

    print("\nFirst 10 split patients missing from this version:")
    display(missing.head(10))

    print("\nFirst 10 outcome disagreements:")
    display(disagreements.head(10))

print("\nComparison finished. No files were modified.")


ENDPOINT VERSION: followup_rfs_audit_all_patients.csv
Rows: 1096
Unique IDs: 1096
Split patients: 357
Matched to this version: 357
Missing from this version: 0
Outcome disagreements among matched: 95

First 10 split patients missing from this version:


,patient_id,survival_time_split,event_split



First 10 outcome disagreements:


,patient_id,file_uuid,years_to_birth,Tumor_purity,pathologic_stage,pathology_T_stage,pathology_N_stage,pathology_M_stage,number_of_lymph_nodes,radiation_therapy,...,ZIC5,CRYAB,EMILIN3,TNNT2,INA,event_split,survival_time_split,survival_time_version,event_version,_merge
0,TCGA-A2-A0SX,64093c9e-aebe-4367-adbb-3ea85e3234c9,-0.809452,-0.491222,1,1,0,0,-0.502090,1,...,-0.128622,1.729647,1.049020,2.234259,1.552731,1,1534.0,NaN,NaN,both
1,TCGA-A2-A3XU,87e7cf96-5ab5-4f26-b839-25cd31607c9c,-1.802022,0.577139,2,2,1,0,0.197181,1,...,2.798723,3.152770,-0.788740,2.693769,-0.879022,1,912.0,574.0,1.0,both
4,TCGA-B6-A0WX,8fef8f57-cb43-43d7-a442-97d095ad203e,-1.420264,-1.325047,3,3,1,0,1.595725,0,...,1.391048,1.613034,1.692524,2.429013,0.959600,1,639.0,563.0,1.0,both
5,TCGA-B6-A0I8,86d7c274-bd7a-4dbb-b2b1-2fdf1033b4dd,-0.962155,0.915996,2,1,0,0,-0.269000,0,...,-0.599734,-0.928972,-0.209136,-0.233601,-0.539572,1,749.0,746.0,1.0,both
6,TCGA-A8-A08O,5f71ff26-3b04-432b-8bfc-1dc439263d79,-1.038506,0.711055,4,2,3,1,-0.269000,0,...,-0.599734,0.079398,-0.044787,-0.800485,-0.455593,1,943.0,NaN,NaN,both
7,TCGA-B6-A0WW,454d3eb9-ff96-4cd4-9dec-17252819ed13,-0.045936,0.758406,2,2,1,0,0.663363,1,...,-0.069914,-0.723435,-1.024589,0.608738,-0.050906,1,558.0,447.0,1.0,both
9,TCGA-E2-A14Z,94dfbf0f-c925-4d1f-804d-f730e7adc29f,0.412173,-1.311730,1,1,0,0,-0.502090,1,...,0.643482,0.111782,-0.024503,-0.448300,-0.057154,1,563.0,188.0,1.0,both
13,TCGA-AR-A24O,4071558e-3697-4cc1-af45-87873ab4e389,-1.191209,-0.815282,3,3,2,0,1.129544,1,...,-0.395686,0.727522,0.685564,-0.350657,-0.102921,1,3607.0,NaN,NaN,both
15,TCGA-A2-A04V,312830d8-fc88-4968-9176-e2240fd4d2d9,-1.496616,0.777642,2,2,0,0,-0.502090,0,...,-0.599734,-0.066937,-0.644190,-0.804987,-0.102824,1,1920.0,NaN,NaN,both
17,TCGA-A2-A25B,611b7039-10a5-4ca9-9498-b5b93531b255,-1.496616,1.011439,2,2,1,0,-0.035909,1,...,-0.599734,-0.517221,-1.401068,-1.043650,1.135936,1,1291.0,1290.0,1.0,both



ENDPOINT VERSION: followup_rfs_audited_v2_candidate.csv
Rows: 1062
Unique IDs: 1062
Split patients: 357
Matched to this version: 337
Missing from this version: 20
Outcome disagreements among matched: 75

First 10 split patients missing from this version:


...


ENDPOINT VERSION: followup_rfs_final_v1.csv
Rows: 1062
Unique IDs: 1062
Split patients: 357
Matched to this version: 337
Missing from this version: 20
Outcome disagreements among matched: 75

First 10 split patients missing from this version:


,patient_id,survival_time_split,event_split
0,TCGA-A2-A0SX,1534.0,1
6,TCGA-A8-A08O,943.0,1
13,TCGA-AR-A24O,3607.0,1
15,TCGA-A2-A04V,1920.0,1
33,TCGA-AR-A0TW,3009.0,1
48,TCGA-A2-A0SW,1365.0,1
77,TCGA-D8-A1XC,377.0,1
83,TCGA-A8-A091,1004.0,1
107,TCGA-A8-A08T,3409.0,1
137,TCGA-A8-A094,1.0,0



First 10 outcome disagreements:


,patient_id,file_uuid,years_to_birth,Tumor_purity,pathologic_stage,pathology_T_stage,pathology_N_stage,pathology_M_stage,number_of_lymph_nodes,radiation_therapy,...,ZIC5,CRYAB,EMILIN3,TNNT2,INA,event_split,survival_time_split,survival_time_version,event_version,_merge
1,TCGA-A2-A3XU,87e7cf96-5ab5-4f26-b839-25cd31607c9c,-1.802022,0.577139,2,2,1,0,0.197181,1,...,2.798723,3.152770,-0.788740,2.693769,-0.879022,1,912.0,574.0,1.0,both
4,TCGA-B6-A0WX,8fef8f57-cb43-43d7-a442-97d095ad203e,-1.420264,-1.325047,3,3,1,0,1.595725,0,...,1.391048,1.613034,1.692524,2.429013,0.959600,1,639.0,563.0,1.0,both
5,TCGA-B6-A0I8,86d7c274-bd7a-4dbb-b2b1-2fdf1033b4dd,-0.962155,0.915996,2,1,0,0,-0.269000,0,...,-0.599734,-0.928972,-0.209136,-0.233601,-0.539572,1,749.0,746.0,1.0,both
7,TCGA-B6-A0WW,454d3eb9-ff96-4cd4-9dec-17252819ed13,-0.045936,0.758406,2,2,1,0,0.663363,1,...,-0.069914,-0.723435,-1.024589,0.608738,-0.050906,1,558.0,447.0,1.0,both
9,TCGA-E2-A14Z,94dfbf0f-c925-4d1f-804d-f730e7adc29f,0.412173,-1.311730,1,1,0,0,-0.502090,1,...,0.643482,0.111782,-0.024503,-0.448300,-0.057154,1,563.0,188.0,1.0,both
17,TCGA-A2-A25B,611b7039-10a5-4ca9-9498-b5b93531b255,-1.496616,1.011439,2,2,1,0,-0.035909,1,...,-0.599734,-0.517221,-1.401068,-1.043650,1.135936,1,1291.0,1290.0,1.0,both
22,TCGA-A2-A04P,471afecc-f0fb-4862-8474-b698feb6f2cb,-1.725670,-0.213774,3,2,3,0,3.693541,1,...,-0.599734,1.792431,0.068443,0.847885,0.787139,1,548.0,102.0,1.0,both
32,TCGA-D8-A73W,7802781b-5141-4f48-ab3c-e8b86899e639,1.557446,0.441745,3,3,1,0,-0.269000,1,...,-0.355805,0.019228,-0.940309,0.159394,-0.002039,1,385.0,21.0,1.0,both
42,TCGA-AO-A0JA,a1ec9d1d-da82-4814-915b-bef1f6a3a3a8,-1.725670,0.451363,3,2,3,0,2.061906,0,...,-0.599734,-0.238153,0.163470,-0.333278,-0.667768,1,655.0,143.0,1.0,both
50,TCGA-BH-A1EY,a8159a6e-5816-493f-9bd0-5061c6d41866,1.557446,-0.726499,2,2,0,0,-0.502090,0,...,-0.599734,-1.023027,-0.810621,-0.288014,-0.403433,1,538.0,396.0,1.0,both



Comparison finished. No files were modified.


In [48]:

from pathlib import Path
import pandas as pd
import numpy as np

RFS_DIR = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint"
)
SPLIT_DIR = Path(
    "/content/drive/MyDrive/TCGA_BRCA/dataset_split_rfs"
)

# Load only the three outcome columns from the fixed split files
split_frames = []

for filename in [
    "train_rfs.csv",
    "validation_rfs.csv",
    "test_rfs.csv",
]:
    df = pd.read_csv(SPLIT_DIR / filename)
    df = df[["patient_id", "survival_time", "event"]].copy()
    split_frames.append(df)

split = pd.concat(split_frames, ignore_index=True)
split["patient_id"] = (
    split["patient_id"].astype("string").str.strip().str.upper()
)
split["survival_time"] = pd.to_numeric(
    split["survival_time"], errors="coerce"
)
split["event"] = pd.to_numeric(split["event"], errors="coerce")

assert split["patient_id"].is_unique, "Duplicate split patient IDs!"

def load_endpoint(filename):
    df = pd.read_csv(RFS_DIR / filename, low_memory=False)
    df = df[["patient_id", "survival_time", "event"]].copy()
    df["patient_id"] = (
        df["patient_id"].astype("string").str.strip().str.upper()
    )
    df["survival_time"] = pd.to_numeric(
        df["survival_time"], errors="coerce"
    )
    df["event"] = pd.to_numeric(df["event"], errors="coerce")
    assert df["patient_id"].is_unique, f"Duplicate IDs in {filename}"
    return df

audit = load_endpoint("followup_rfs_audit_all_patients.csv")
candidate = load_endpoint("followup_rfs_audited_v2_candidate.csv")
final_v1 = load_endpoint("followup_rfs_final_v1.csv")

# 1. Inspect the 20 split patients absent from the candidate endpoint
candidate_ids = set(candidate["patient_id"].dropna())
missing_ids = sorted(set(split["patient_id"]) - candidate_ids)

print("Split patients absent from candidate:", len(missing_ids))

missing_check = (
    split[split["patient_id"].isin(missing_ids)]
    .merge(
        audit,
        on="patient_id",
        how="left",
        suffixes=("_split", "_audit"),
        validate="one_to_one",
        indicator=True,
    )
)

print("\nOUTCOMES FOR PATIENTS ABSENT FROM THE CANDIDATE")
display(missing_check)

# 2. Compare the older audit against the candidate for shared patients
audit_candidate = audit.merge(
    candidate,
    on="patient_id",
    how="inner",
    suffixes=("_audit", "_candidate"),
    validate="one_to_one",
)

time_equal = np.isclose(
    audit_candidate["survival_time_audit"],
    audit_candidate["survival_time_candidate"],
    equal_nan=False,
)
event_equal = (
    audit_candidate["event_audit"].to_numpy()
    == audit_candidate["event_candidate"].to_numpy()
)

print("\nAUDIT vs CANDIDATE")
print("Shared patients:", len(audit_candidate))
print("Survival-time disagreements:", int((~time_equal).sum()))
print("Event disagreements:", int((~event_equal).sum()))
print(
    "Any outcome disagreement:",
    int((~time_equal | ~event_equal).sum())
)

print("\nFIRST 20 OUTCOME DIFFERENCES")
display(
    audit_candidate.loc[
        (~time_equal) | (~event_equal),
        [
            "patient_id",
            "survival_time_audit",
            "event_audit",
            "survival_time_candidate",
            "event_candidate",
        ],
    ].head(20)
)

# 3. Check whether candidate and final_v1 are identical
candidate_final = candidate.merge(
    final_v1,
    on="patient_id",
    how="outer",
    suffixes=("_candidate", "_final"),
    indicator=True,
    validate="one_to_one",
)

same_ids = candidate_final["_merge"].eq("both")
same_times = np.isclose(
    candidate_final["survival_time_candidate"],
    candidate_final["survival_time_final"],
    equal_nan=False,
)
same_events = (
    candidate_final["event_candidate"].to_numpy()
    == candidate_final["event_final"].to_numpy()
)

print("\nCANDIDATE vs FINAL_V1")
print("Rows present in both:", int(same_ids.sum()))
print("Rows missing from either version:", int((~same_ids).sum()))
print(
    "Outcome differences among shared rows:",
    int((same_ids & (~same_times | ~same_events)).sum())
)

print("\nDiagnostic complete. No files were modified.")


Split patients absent from candidate: 20

OUTCOMES FOR PATIENTS ABSENT FROM THE CANDIDATE


,patient_id,survival_time_split,event_split,survival_time_audit,event_audit,_merge
0,TCGA-A2-A0SX,1534.0,1,NaN,NaN,both
1,TCGA-A8-A08O,943.0,1,NaN,NaN,both
2,TCGA-AR-A24O,3607.0,1,NaN,NaN,both
3,TCGA-A2-A04V,1920.0,1,NaN,NaN,both
4,TCGA-AR-A0TW,3009.0,1,NaN,NaN,both
5,TCGA-A2-A0SW,1365.0,1,NaN,NaN,both
6,TCGA-D8-A1XC,377.0,1,NaN,NaN,both
7,TCGA-A8-A091,1004.0,1,NaN,NaN,both
8,TCGA-A8-A08T,3409.0,1,NaN,NaN,both
9,TCGA-A8-A094,1.0,0,NaN,NaN,both



AUDIT vs CANDIDATE
Shared patients: 1062
Survival-time disagreements: 0
Event disagreements: 0
Any outcome disagreement: 0

FIRST 20 OUTCOME DIFFERENCES


,patient_id,survival_time_audit,event_audit,survival_time_candidate,event_candidate



CANDIDATE vs FINAL_V1
Rows present in both: 1062
Rows missing from either version: 0
Outcome differences among shared rows: 0

Diagnostic complete. No files were modified.
